# Generación de entregables de telecomunicaciones en Sonora

Esta libreta documenta el proceso metodológico utilizado para transformar los archivos fuente `.txt` y el Modelo Digital de Elevación del INEGI en los datasets finales del proyecto:

- `Dataset_Instituciones_Unificadas.csv`
- `dataset_instituciones_cercanas_a_torres_estatales.csv`
- `dataset_instituciones_sin_torre_estatal_cercana.csv`
- `migraciones_posibles.csv`
- `conexiones_nivel3_torres_estatales.csv`
- `torres_propuestas_actualizado.csv`

El objetivo del análisis es optimizar la infraestructura pública de telecomunicaciones del Estado de Sonora. Para ello se integran instituciones públicas, se calcula su cercanía a torres estatales, se clasifica su prioridad mediante el Índice de Prioridad de Conectividad (IPC), se identifican instituciones candidatas a migración, se separan instituciones sin conexión que pueden aprovechar torres estatales existentes y se proponen zonas para nueva infraestructura.


## Metodología resumida

1. Leer los archivos fuente en formato JSON/GeoJSON almacenados como `.txt`.
2. Normalizar instituciones de cuatro sectores: escuelas, salud, gobierno y seguridad.
3. Normalizar torres estatales como infraestructura de referencia.
4. Calcular la torre estatal más cercana usando distancia Haversine.
5. Clasificar instituciones en cuatro niveles del IPC:
   - **Nivel 1:** institución conectada mediante proveedor público o estatal.
   - **Nivel 2:** institución conectada mediante proveedor privado.
   - **Nivel 3:** institución sin conectividad, pero con torre estatal cercana.
   - **Nivel 4:** institución sin conectividad y sin torre estatal cercana.
6. Generar candidatos a migración para instituciones de Nivel 2 cercanas a torres estatales.
7. Generar candidatos a conexión inicial para instituciones de Nivel 3 cercanas a torres estatales.
8. Agrupar instituciones de Nivel 4 con `DBSCAN` para proponer nueva infraestructura.
9. Validar línea de vista entre instituciones/torres propuestas y torres estatales usando el Modelo Digital de Elevación del INEGI.

> Para que la libreta pueda ejecutarse en Google Colab, se incluyen enlaces de descarga con `gdown` para los `.txt` y el raster `.tif` de elevación.


In [6]:
# Si el ambiente no tiene estas librerías, ejecutar una vez:
# %pip install pandas scikit-learn gdown rasterio

from pathlib import Path
import json
import math
import os
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import haversine_distances

In [7]:
# Rutas de trabajo
BASE_DIR = Path(".")
OUTPUT_DIR = BASE_DIR / "ENTREGABLES_FINALES"
OUTPUT_DIR.mkdir(exist_ok=True)

FUENTES_INSTITUCIONES = {
    "ESCUELAS": BASE_DIR / "Escuelas.txt",
    "GOBIERNO": BASE_DIR / "Gobierno.txt",
    "SALUD": BASE_DIR / "Salud.txt",
    "SEGURIDAD": BASE_DIR / "Seguridad.txt",
}

FUENTE_TORRES = BASE_DIR / "INFORMACION TORRES.txt"
FUENTE_MDE = BASE_DIR / "26_Sonora_r15m_v4.tif"

# Parámetros principales del análisis
RADIO_TORRE_KM = 10
RADIO_CLUSTER_KM = 10
MIN_INSTITUCIONES_CLUSTER = 5
EARTH_RADIUS_KM = 6371.0088

# Parámetros usados para reproducir los criterios técnicos del análisis.
# Las migraciones se calculan directamente desde los .txt con reglas visibles y reproducibles.
ALTURA_TORRE_PROPUESTA_M = 30
ALTURA_TORRE_ESTATAL_M = 30
ALTURA_SITIO_INSTITUCION_M = 10
MUESTRAS_LINEA_VISTA = 120
MARGEN_OBSTRUCCION_M = 0

## Descarga

Esta celda permite correr la libreta en Colab sin subir archivos manualmente. Si los archivos ya existen en la carpeta de trabajo, no los vuelve a descargar.

In [8]:
ARCHIVOS_GDOWN = {
    "26_Sonora_r15m_v4.tif": "1yRPUtJs8OZvtp8yIr8Nid_VbPNcvzcJ9",
    "Escuelas.txt": "1i6-x4Tf0pIp5uqmfB8UDjzyEzlt_YB0i",
    "INFORMACION TORRES.txt": "1J28fP0I-3u6vJYgnF99v3VcoonX3v7QD",
    "Gobierno.txt": "1Nmy8g7L5PelBIgVNlbqMniQChZDXLT7h",
    "Salud.txt": "18djIzdZIg4uqN5obiDTnHSIbgefqH4oG",
    "Seguridad.txt": "1TosTl-IVqQ554ajihoXWT6TFFJfHtAtW",
}


def descargar(archivos=ARCHIVOS_GDOWN):
    faltantes = {nombre: file_id for nombre, file_id in archivos.items() if not (BASE_DIR / nombre).exists()}
    if not faltantes:
        print("Ya están disponibles en la carpeta de trabajo.")
        return

    try:
        import gdown
    except ImportError as exc:
        raise ImportError("Instala gdown antes de descargar: %pip install gdown") from exc

    for nombre, file_id in faltantes.items():
        destino = BASE_DIR / nombre
        url = f"https://drive.google.com/uc?id={file_id}"
        print(f"Descargando {nombre}...")
        gdown.download(url, str(destino), quiet=False)


descargar()

Ya están disponibles en la carpeta de trabajo.


## Funciones auxiliares

Estas funciones corrigen diferencias de escritura entre archivos, limpian texto, interpretan campos booleanos y calculan distancias geográficas.

In [4]:
def quitar_acentos(texto):
    texto = "" if pd.isna(texto) else str(texto)
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def normalizar_texto(texto):
    texto = quitar_acentos(texto).upper().strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto


def texto_limpio(texto):
    if texto is None or pd.isna(texto):
        return ""
    return str(texto).strip()


def a_float(valor):
    if valor is None or valor == "" or valor == "-":
        return np.nan
    try:
        return float(valor)
    except (TypeError, ValueError):
        return np.nan


def corregir_coordenadas_sonora(lat, lon):
    """Corrige longitudes con signo positivo cuando el punto corresponde a Sonora."""
    lat = a_float(lat)
    lon = a_float(lon)
    if 26 <= lat <= 33 and 108 <= lon <= 116:
        lon = -lon
    return lat, lon


def conectividad_normalizada(valor):
    valor_norm = normalizar_texto(valor)
    if valor_norm in {"TRUE", "CON CONECTIVIDAD", "CONECTADA", "SI", "SÍ"}:
        return "CONECTADA"
    return "SIN CONEXION"


def distancia_haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(a))


PROVEEDORES_PUBLICOS = {"CFE", "TELEFONIA RURAL"}


def es_proveedor_publico(nombre):
    nombre_norm = normalizar_texto(nombre)
    return any(proveedor in nombre_norm for proveedor in PROVEEDORES_PUBLICOS)


def resumir_proveedores(proveedores):
    proveedores = proveedores if isinstance(proveedores, list) else []
    nombres = [texto_limpio(p.get("nombre proveedor")) for p in proveedores if p.get("nombre proveedor")]
    nombres_unicos = sorted(set(nombres))
    tiene_publico = any(es_proveedor_publico(nombre) for nombre in nombres_unicos)
    tiene_privado = any(not es_proveedor_publico(nombre) for nombre in nombres_unicos)

    proveedor_privado = ""
    costo_privado = ""
    for proveedor in proveedores:
        nombre = texto_limpio(proveedor.get("nombre proveedor"))
        if nombre and not es_proveedor_publico(nombre):
            proveedor_privado = nombre
            costo = proveedor.get("costo", "")
            costo_privado = "" if costo in [None, "-"] else costo
            break

    return {
        "proveedores_nombres": "; ".join(nombres_unicos),
        "tiene_proveedor_publico": int(tiene_publico),
        "tiene_proveedor_privado": int(tiene_privado),
        "proveedor_privado": proveedor_privado,
        "costo_privado": costo_privado,
    }

## Lectura y normalización de fuentes

Los archivos `.txt` contienen listas de objetos con una estructura tipo GeoJSON: cada registro tiene propiedades descriptivas y coordenadas. Aquí convertimos esos registros en tablas de instituciones y torres.

In [5]:
def leer_json_txt(ruta):
    with open(ruta, "r", encoding="utf-8-sig") as f:
        return json.load(f)


def normalizar_instituciones():
    registros = []

    for grupo_general, ruta in FUENTES_INSTITUCIONES.items():
        data = leer_json_txt(ruta)
        for item in data:
            geojson = item.get("geojson", {})
            props = geojson.get("properties", {})
            coords = geojson.get("geometry", {}).get("coordinates", [np.nan, np.nan])
            lon, lat = coords[0], coords[1]
            lat, lon = corregir_coordenadas_sonora(lat, lon)
            proveedores = props.get("proveedores", [])
            resumen_prov = resumir_proveedores(proveedores)
            conectividad = conectividad_normalizada(props.get("conectividad"))

            nombre = (
                props.get("nombre")
                or props.get("unidad/organizacion")
                or props.get("dependencia")
                or props.get("clave unica")
                or item.get("id")
            )

            registros.append({
                "ID": str(item.get("id")),
                "CLAVE_UNICA": texto_limpio(props.get("clave unica")),
                "GRUPO_INSTITUCION": grupo_general,
                "grupo": normalizar_texto(props.get("grupo") or grupo_general).lower(),
                "NOMBRE": texto_limpio(nombre),
                "MUNICIPIO": normalizar_texto(props.get("municipio")),
                "LOCALIDAD": normalizar_texto(props.get("localidad")),
                "LATITUD": lat,
                "LONGITUD": lon,
                "CONECTIVIDAD": conectividad,
                **resumen_prov,
            })

    instituciones = pd.DataFrame(registros)
    instituciones = instituciones.dropna(subset=["LATITUD", "LONGITUD"]).reset_index(drop=True)
    return instituciones


def normalizar_torres():
    registros = []
    data = leer_json_txt(FUENTE_TORRES)

    for item in data:
        geojson = item.get("geojson", {})
        props = geojson.get("properties", {})
        coords = geojson.get("geometry", {}).get("coordinates", [np.nan, np.nan])
        lon, lat = coords[0], coords[1]
        lat, lon = corregir_coordenadas_sonora(lat, lon)

        registros.append({
            "ID_TORRE": str(item.get("id")),
            "NOMBRE_TORRE": texto_limpio(props.get("nombre")),
            "MUNICIPIO_TORRE": normalizar_texto(props.get("municipio")),
            "DEPENDENCIA_TORRE": texto_limpio(
                props.get("dependencia administradora")
                or props.get("propietario")
                or props.get("arrendador")
            ),
            "LATITUD_TORRE": lat,
            "LONGITUD_TORRE": lon,
        })

    torres = pd.DataFrame(registros)
    torres = torres.dropna(subset=["LATITUD_TORRE", "LONGITUD_TORRE"]).reset_index(drop=True)
    return torres


instituciones = normalizar_instituciones()
torres = normalizar_torres()

print("Instituciones cargadas:", len(instituciones))
print("Torres cargadas:", len(torres))
display(instituciones.head())
display(torres.head())

Instituciones cargadas: 3547
Torres cargadas: 466


,ID,CLAVE_UNICA,GRUPO_INSTITUCION,grupo,NOMBRE,MUNICIPIO,LOCALIDAD,LATITUD,LONGITUD,CONECTIVIDAD,proveedores_nombres,tiene_proveedor_publico,tiene_proveedor_privado,proveedor_privado,costo_privado
0,1,26DJN0412P,ESCUELAS,escuelas,SILVIANA ROBLES DE ROMERO,ACONCHI,LA ESTANCIA,29.793333,-110.211389,SIN CONEXION,,0,0,,
1,2,26DPR0306W,ESCUELAS,escuelas,GENERAL FRANCISCO CONTRERAS B,ACONCHI,LA ESTANCIA,29.793333,-110.211389,CONECTADA,TELMEX,0,1,TELMEX,
2,109,26DPR0283B,ESCUELAS,escuelas,BENITO JUAREZ,ALAMOS,21 DE MARZO,26.527222,-109.073889,CONECTADA,INDEX,0,1,INDEX,
3,4,26EJN0005I,ESCUELAS,escuelas,BELEM M. DE GANDARA,ACONCHI,ACONCHI,29.825733,-110.227842,CONECTADA,TELMEX,0,1,TELMEX,
4,519,26DPR1011R,ESCUELAS,escuelas,AQUILES SERDAN,BACUM,ATOTONILCO,27.306667,-110.171389,CONECTADA,INDEX,0,1,INDEX,


,ID_TORRE,NOMBRE_TORRE,MUNICIPIO_TORRE,DEPENDENCIA_TORRE,LATITUD_TORRE,LONGITUD_TORRE
0,4357,Torre Cerro Aguilas,IMURIS,Seguridad,30.744739,-110.760278
1,4305,Siete Cerros,HERMOSILLO,Seguridad,28.857243,-111.386448
2,4306,(TZ-90) Comandancia Municipal,HERMOSILLO,Seguridad,28.857675,-112.013917
3,4307,ADIVINO,VILLA PESQUEIRA,Telemax,28.977411,-110.038236
4,4308,BACANORA,BACANORA,Telemax,28.977583,-109.399342


## Cálculo de torre estatal más cercana

Para calcular cercanía entre instituciones y torres se usa `BallTree` con métrica Haversine. Este método trabaja con coordenadas en radianes y permite encontrar eficientemente la torre más cercana para cada institución.

In [6]:
def agregar_torre_mas_cercana(instituciones, torres):
    inst = instituciones.copy()
    tw = torres.copy()

    coords_torres_rad = np.radians(tw[["LATITUD_TORRE", "LONGITUD_TORRE"]].to_numpy())
    coords_inst_rad = np.radians(inst[["LATITUD", "LONGITUD"]].to_numpy())

    tree = BallTree(coords_torres_rad, metric="haversine")
    dist_rad, idx = tree.query(coords_inst_rad, k=1)

    inst["DISTANCIA_TORRE_ESTATAL_CERCANA"] = dist_rad[:, 0] * EARTH_RADIUS_KM
    inst["INDICE_TORRE_CERCANA"] = idx[:, 0]

    torres_cercanas = tw.iloc[idx[:, 0]].reset_index(drop=True)
    inst["TORRE_CERCANA"] = torres_cercanas["NOMBRE_TORRE"].values
    inst["DEPENDENCIA_TORRE_ESTATAL"] = torres_cercanas["DEPENDENCIA_TORRE"].values
    inst["LATITUD_TORRE_ESTATAL"] = torres_cercanas["LATITUD_TORRE"].values
    inst["LONGITUD_TORRE_ESTATAL"] = torres_cercanas["LONGITUD_TORRE"].values
    inst["TORRE_CERCANA_COORD"] = (
        inst["LATITUD_TORRE_ESTATAL"].round(6).astype(str)
        + ", "
        + inst["LONGITUD_TORRE_ESTATAL"].round(6).astype(str)
    )
    inst["torre_cercana"] = (inst["DISTANCIA_TORRE_ESTATAL_CERCANA"] <= RADIO_TORRE_KM).astype(int)
    return inst


instituciones_geo = agregar_torre_mas_cercana(instituciones, torres)

print("Distancia promedio a torre estatal:", round(instituciones_geo["DISTANCIA_TORRE_ESTATAL_CERCANA"].mean(), 2), "km")
display(instituciones_geo[["ID", "GRUPO_INSTITUCION", "NOMBRE", "DISTANCIA_TORRE_ESTATAL_CERCANA", "TORRE_CERCANA"]].head())

Distancia promedio a torre estatal:

 3.9 km


,ID,GRUPO_INSTITUCION,NOMBRE,DISTANCIA_TORRE_ESTATAL_CERCANA,TORRE_CERCANA
0,1,ESCUELAS,SILVIANA ROBLES DE ROMERO,3.745554,ACONCHI (SITIO ALTAN)
1,2,ESCUELAS,GENERAL FRANCISCO CONTRERAS B,3.745554,ACONCHI (SITIO ALTAN)
2,109,ESCUELAS,BENITO JUAREZ,0.219759,Prim Benito Juárez de 21 de Marzo
3,4,ESCUELAS,BELEM M. DE GANDARA,0.768855,ACONCHI (SITIO ALTAN)
4,519,ESCUELAS,AQUILES SERDAN,0.158739,Prim Aquiles Serdán de Atotonilco


## Clasificación con el Índice de Prioridad de Conectividad

El IPC resume la situación de conectividad de cada institución y la estrategia operativa más natural:

- **Nivel 1 (bajo):** ya cuenta con proveedor público o estatal.
- **Nivel 2 (medio):** cuenta con conectividad, pero depende de proveedor privado.
- **Nivel 3 (alto):** no cuenta con conectividad, pero tiene una torre estatal a 10 km o menos.
- **Nivel 4 (crítico):** no cuenta con conectividad y está a más de 10 km de una torre estatal.


In [7]:
def asignar_ipc(row):
    if row["tiene_proveedor_publico"] == 1:
        return "NIVEL 1 (BAJO)"
    if row["CONECTIVIDAD"] == "CONECTADA":
        return "NIVEL 2 (MEDIO)"
    if row["DISTANCIA_TORRE_ESTATAL_CERCANA"] <= RADIO_TORRE_KM:
        return "NIVEL 3 (ALTO)"
    return "NIVEL 4 (CRITICO)"


instituciones_geo["PROVEEDOR_PUBLICO"] = instituciones_geo["tiene_proveedor_publico"]
instituciones_geo["IPC"] = instituciones_geo.apply(asignar_ipc, axis=1)

resumen_ipc = (
    instituciones_geo
    .groupby(["GRUPO_INSTITUCION", "IPC"])
    .size()
    .unstack(fill_value=0)
)

display(resumen_ipc)
print("Total de instituciones evaluadas:", len(instituciones_geo))

IPC,NIVEL 1 (BAJO),NIVEL 2 (MEDIO),NIVEL 3 (ALTO),NIVEL 4 (CRITICO)
GRUPO_INSTITUCION,,,,
ESCUELAS,715,1505,472,162
GOBIERNO,0,205,136,2
SALUD,0,110,147,44
SEGURIDAD,20,29,0,0


Total de instituciones evaluadas: 3547


## Entregable 1: Dataset unificado

Este archivo consolida el ecosistema de instituciones públicas e incluye coordenadas, conectividad, distancia a infraestructura estatal, proveedor público e IPC.

In [8]:
columnas_unificado = [
    "ID",
    "GRUPO_INSTITUCION",
    "NOMBRE",
    "MUNICIPIO",
    "LOCALIDAD",
    "LATITUD",
    "LONGITUD",
    "CONECTIVIDAD",
    "DISTANCIA_TORRE_ESTATAL_CERCANA",
    "TORRE_CERCANA_COORD",
    "PROVEEDOR_PUBLICO",
    "IPC",
]

dataset_unificado = instituciones_geo[columnas_unificado].copy()
dataset_unificado["LATITUD"] = dataset_unificado["LATITUD"].round(8)
dataset_unificado["LONGITUD"] = dataset_unificado["LONGITUD"].round(8)
dataset_unificado["DISTANCIA_TORRE_ESTATAL_CERCANA"] = dataset_unificado["DISTANCIA_TORRE_ESTATAL_CERCANA"].round(2)

ruta_unificado = OUTPUT_DIR / "Dataset_Instituciones_Unificadas.csv"
dataset_unificado.to_csv(ruta_unificado, index=False, encoding="utf-8-sig")

print("Archivo generado:", ruta_unificado)
print("Filas:", len(dataset_unificado), "Columnas:", len(dataset_unificado.columns))
display(dataset_unificado.head())

Archivo generado: ENTREGABLES_FINALES\Dataset_Instituciones_Unificadas.csv
Filas: 3547 Columnas: 12


,ID,GRUPO_INSTITUCION,NOMBRE,MUNICIPIO,LOCALIDAD,LATITUD,LONGITUD,CONECTIVIDAD,DISTANCIA_TORRE_ESTATAL_CERCANA,TORRE_CERCANA_COORD,PROVEEDOR_PUBLICO,IPC
0,1,ESCUELAS,SILVIANA ROBLES DE ROMERO,ACONCHI,LA ESTANCIA,29.793333,-110.211389,SIN CONEXION,3.75,"29.8262, -110.21989",0,NIVEL 3 (ALTO)
1,2,ESCUELAS,GENERAL FRANCISCO CONTRERAS B,ACONCHI,LA ESTANCIA,29.793333,-110.211389,CONECTADA,3.75,"29.8262, -110.21989",0,NIVEL 2 (MEDIO)
2,109,ESCUELAS,BENITO JUAREZ,ALAMOS,21 DE MARZO,26.527222,-109.073889,CONECTADA,0.22,"26.525942, -109.075572",0,NIVEL 2 (MEDIO)
3,4,ESCUELAS,BELEM M. DE GANDARA,ACONCHI,ACONCHI,29.825733,-110.227842,CONECTADA,0.77,"29.8262, -110.21989",0,NIVEL 2 (MEDIO)
4,519,ESCUELAS,AQUILES SERDAN,BACUM,ATOTONILCO,27.306667,-110.171389,CONECTADA,0.16,"27.305907, -110.172749",0,NIVEL 2 (MEDIO)


## Entregables 2 y 3: Instituciones cercanas y sin torre estatal cercana

Se crean dos subconjuntos operativos:

- Instituciones a **10 km o menos** de una torre estatal.
- Instituciones a **más de 10 km** de una torre estatal.

El segundo archivo contiene el universo completo de instituciones ubicadas a más de 10 km de una torre estatal. La propuesta de nuevas torres se calcula después mediante agrupamiento territorial, sin fijar manualmente una cantidad de puntos.


In [9]:
columnas_segmentacion = [
    "grupo",
    "ID",
    "CLAVE_UNICA",
    "NOMBRE",
    "torre_cercana",
    "tiene_proveedor_publico",
    "tiene_proveedor_privado",
    "DISTANCIA_TORRE_ESTATAL_CERCANA",
]

segmentacion = instituciones_geo[columnas_segmentacion].copy()
segmentacion["id"] = segmentacion["CLAVE_UNICA"].where(segmentacion["CLAVE_UNICA"] != "", segmentacion["ID"])
segmentacion = segmentacion.rename(columns={
    "NOMBRE": "nombre",
    "DISTANCIA_TORRE_ESTATAL_CERCANA": "dist_km_torre_estatal_mas_cercana",
})

segmentacion_export = segmentacion[[
    "grupo",
    "id",
    "nombre",
    "torre_cercana",
    "tiene_proveedor_publico",
    "tiene_proveedor_privado",
    "dist_km_torre_estatal_mas_cercana",
]].copy()

cercanas = segmentacion_export[
    segmentacion_export["dist_km_torre_estatal_mas_cercana"] <= RADIO_TORRE_KM
].copy()

sin_torre = segmentacion_export[
    segmentacion_export["dist_km_torre_estatal_mas_cercana"] > RADIO_TORRE_KM
].copy()
sin_torre = sin_torre.sort_values("dist_km_torre_estatal_mas_cercana", ascending=False)

ruta_cercanas = OUTPUT_DIR / "dataset_instituciones_cercanas_a_torres_estatales.csv"
ruta_sin_torre = OUTPUT_DIR / "dataset_instituciones_sin_torre_estatal_cercana.csv"

cercanas.to_csv(ruta_cercanas, index=False, encoding="utf-8-sig")
sin_torre.to_csv(ruta_sin_torre, index=False, encoding="utf-8-sig")

print("Archivo generado:", ruta_cercanas, "| filas:", len(cercanas))
print("Archivo generado:", ruta_sin_torre, "| filas:", len(sin_torre))
display(cercanas.head())
display(sin_torre.head())

Archivo generado: ENTREGABLES_FINALES\dataset_instituciones_cercanas_a_torres_estatales.csv | filas: 3140
Archivo generado: ENTREGABLES_FINALES\dataset_instituciones_sin_torre_estatal_cercana.csv | filas: 407


,grupo,id,nombre,torre_cercana,tiene_proveedor_publico,tiene_proveedor_privado,dist_km_torre_estatal_mas_cercana
0,escuelas,26DJN0412P,SILVIANA ROBLES DE ROMERO,1,0,0,3.745554
1,escuelas,26DPR0306W,GENERAL FRANCISCO CONTRERAS B,1,0,1,3.745554
2,escuelas,26DPR0283B,BENITO JUAREZ,1,0,1,0.219759
3,escuelas,26EJN0005I,BELEM M. DE GANDARA,1,0,1,0.768855
4,escuelas,26DPR1011R,AQUILES SERDAN,1,0,1,0.158739


,grupo,id,nombre,torre_cercana,tiene_proveedor_publico,tiene_proveedor_privado,dist_km_torre_estatal_mas_cercana
1915,escuelas,26DPR0670U,JUAN GUTIERREZ NEYOI,0,0,0,53.561007
1913,escuelas,26DJN0160B,LIBERTAD,0,0,0,52.897910
1129,escuelas,26DST0058Q,FUNDACION ESPOSOS RODRIGUEZ,0,0,1,52.818078
2805,escuelas,26EMS0018E,"EMSaD, Puerto Libertad",0,0,1,52.770558
2403,escuelas,26DPR1072E,RUBEN C NAVARRO,0,0,1,52.729717


## Entregable 4: Migraciones posibles

Las instituciones candidatas a migración cumplen cuatro condiciones:

1. Están clasificadas como **Nivel 2**.
2. Tienen proveedor privado.
3. Se ubican a **10 km o menos** de una torre estatal.
4. Se valida su línea de vista con la torre estatal más cercana usando el MDE de INEGI.

Este entregable representa una estrategia de **optimización del gasto operativo**: las instituciones ya cuentan con conectividad, pero dependen de proveedores privados. Por eso se consideran candidatas a migrar hacia infraestructura estatal cuando existe una torre pública cercana.

Estas instituciones no están conectadas actualmente a una torre estatal; están conectadas con proveedores privados y se consideran candidatas porque existe infraestructura pública cercana. Para esta validación se usa una altura estimada de 10 m en el sitio de la institución y 30 m en la torre estatal.


In [10]:
def validar_lineas_vista_con_mde(puntos_df, raster_path=FUENTE_MDE):
    """Valida línea de vista entre un punto origen y su torre estatal más cercana."""
    if not Path(raster_path).exists():
        raise FileNotFoundError(f"No se encontró el MDE: {raster_path}")

    try:
        import rasterio
        proj_data = Path(rasterio.__file__).parent / "proj_data"
        if proj_data.exists():
            os.environ["PROJ_DATA"] = str(proj_data)
            os.environ["PROJ_LIB"] = str(proj_data)
        from rasterio.warp import transform as transformar_coords
    except ImportError as exc:
        raise ImportError("Instala rasterio antes de validar línea de vista: %pip install rasterio") from exc

    resultados = []

    with rasterio.open(raster_path) as mde:
        nodata = mde.nodata

        for _, row in puntos_df.iterrows():
            lats = np.linspace(row["LATITUD"], row["LATITUD_TORRE_ESTATAL"], MUESTRAS_LINEA_VISTA)
            lons = np.linspace(row["LONGITUD"], row["LONGITUD_TORRE_ESTATAL"], MUESTRAS_LINEA_VISTA)

            xs, ys = transformar_coords("EPSG:4326", mde.crs, lons.tolist(), lats.tolist())
            muestras = np.array([valor[0] for valor in mde.sample(zip(xs, ys))], dtype="float64")

            if nodata is not None:
                muestras = np.where(muestras == nodata, np.nan, muestras)

            if np.isnan(muestras).any():
                resultados.append({
                    "LINEA_VISTA": False,
                    "OBSTRUCCION_MAX_M": np.nan,
                    "ELEVACION_ORIGEN_M": np.nan,
                    "ELEVACION_TORRE_ESTATAL_M": np.nan,
                })
                continue

            altura_inicio = muestras[0] + ALTURA_SITIO_INSTITUCION_M
            altura_fin = muestras[-1] + ALTURA_TORRE_ESTATAL_M
            linea_senal = np.linspace(altura_inicio, altura_fin, MUESTRAS_LINEA_VISTA)

            # Se excluyen los extremos porque corresponden al punto origen y a la torre.
            obstrucciones = muestras[1:-1] - linea_senal[1:-1] + MARGEN_OBSTRUCCION_M
            obstruccion_max = float(np.max(obstrucciones)) if len(obstrucciones) else 0.0
            linea_vista = obstruccion_max <= 0

            resultados.append({
                "LINEA_VISTA": bool(linea_vista),
                "OBSTRUCCION_MAX_M": obstruccion_max,
                "ELEVACION_ORIGEN_M": float(muestras[0]),
                "ELEVACION_TORRE_ESTATAL_M": float(muestras[-1]),
            })

    resultados_df = pd.DataFrame(resultados)
    return pd.concat([puntos_df.reset_index(drop=True), resultados_df], axis=1)


instituciones_geo["POSIBLE_MIGRAR_TORRE_ESTATAL"] = (
    (instituciones_geo["DISTANCIA_TORRE_ESTATAL_CERCANA"] <= RADIO_TORRE_KM)
    & (instituciones_geo["TORRE_CERCANA"].fillna("") != "")
)

migraciones_candidatas = instituciones_geo[
    (instituciones_geo["IPC"] == "NIVEL 2 (MEDIO)")
    & (instituciones_geo["tiene_proveedor_privado"] == 1)
    & (instituciones_geo["POSIBLE_MIGRAR_TORRE_ESTATAL"])
].copy()

migraciones_candidatas = migraciones_candidatas.sort_values("DISTANCIA_TORRE_ESTATAL_CERCANA", ascending=True)
migraciones_candidatas = validar_lineas_vista_con_mde(migraciones_candidatas)
migraciones_candidatas["CONEXION_MIGRACION"] = np.where(
    migraciones_candidatas["LINEA_VISTA"],
    "MIGRACION DIRECTA VIABLE",
    "REQUIERE REPETIDOR O REVISION TECNICA",
)

migraciones_export = pd.DataFrame({
    "ID_INSTITUCION": migraciones_candidatas["ID"],
    "INSTITUCION": migraciones_candidatas["NOMBRE"],
    "PROVEEDOR": migraciones_candidatas["proveedor_privado"],
    "COSTO": migraciones_candidatas["costo_privado"],
    "TORRE_CERCANA": migraciones_candidatas["TORRE_CERCANA"],
    "LATITUD_TORRE": migraciones_candidatas["LATITUD_TORRE_ESTATAL"].round(6),
    "LONGITUD_TORRE": migraciones_candidatas["LONGITUD_TORRE_ESTATAL"].round(6),
    "DIST_KM": migraciones_candidatas["DISTANCIA_TORRE_ESTATAL_CERCANA"].round(3),
    "LINEA_VISTA": migraciones_candidatas["LINEA_VISTA"],
    "CONEXION_MIGRACION": migraciones_candidatas["CONEXION_MIGRACION"],
    "OBSTRUCCION_MAX_M": migraciones_candidatas["OBSTRUCCION_MAX_M"].round(2),
})

ruta_migraciones = OUTPUT_DIR / "migraciones_posibles.csv"
migraciones_export.to_csv(ruta_migraciones, index=False, encoding="utf-8-sig")

print("Archivo generado:", ruta_migraciones)
print("Instituciones candidatas a migración:", len(migraciones_export))
print("Candidatas con línea de vista directa:", int(migraciones_export["LINEA_VISTA"].sum()))
display(migraciones_export.head())

Archivo generado: ENTREGABLES_FINALES\migraciones_posibles.csv
Instituciones candidatas a migración: 1713
Candidatas con línea de vista directa: 1642


,ID_INSTITUCION,INSTITUCION,PROVEEDOR,COSTO,TORRE_CERCANA,LATITUD_TORRE,LONGITUD_TORRE,DIST_KM,LINEA_VISTA,CONEXION_MIGRACION,OBSTRUCCION_MAX_M
0,3563,Universidad de la Sierra,TELMEX,,Universidad de la Sierra de Moctezuma,29.823617,-109.678510,0.000,True,MIGRACION DIRECTA VIABLE,-10.17
1,2981,Plantel Plutarco Elías Calles (Agua Prieta),TELMEX,,PLANTEL PLUTARCO ELÍAS CALLES,31.315724,-109.555035,0.000,True,MIGRACION DIRECTA VIABLE,-10.17
2,3226,HOSPITAL GENERAL DE OBREGÓN,MEGACABLE,,Hospital General de Obregón,27.483600,-109.955000,0.003,True,MIGRACION DIRECTA VIABLE,-10.17
3,1399,RICARDO MONTIJO CASTRO,TELMEX,,NACO,31.327333,-109.948686,0.003,True,MIGRACION DIRECTA VIABLE,-10.17
4,3660,CECAP,TELMEX,1292.24,Oficina de Telefonia Rural de Sonora (Edificio...,29.067570,-110.958450,0.004,True,MIGRACION DIRECTA VIABLE,-10.17


## Entregable 5: Conexiones Nivel 3 mediante torres estatales existentes

Las instituciones de **Nivel 3** no se tratan como migraciones, porque no tienen una conexión activa que sustituir. Se tratan como candidatas a **conexión inicial** usando infraestructura estatal existente.

El criterio reproducible es:

1. Institución clasificada como **Nivel 3**.
2. Sin conectividad registrada en los archivos fuente.
3. Torre estatal a **10 km o menos**.
4. Validación de línea de vista con el Modelo Digital de Elevación del INEGI.

Esta separación permite distinguir dos estrategias distintas: Nivel 2 optimiza costos mediante migración; Nivel 3 amplía cobertura usando torres estatales ya existentes.


In [11]:
nivel3_candidatas = instituciones_geo[
    (instituciones_geo["IPC"] == "NIVEL 3 (ALTO)")
    & (instituciones_geo["CONECTIVIDAD"] == "SIN CONEXION")
    & (instituciones_geo["DISTANCIA_TORRE_ESTATAL_CERCANA"] <= RADIO_TORRE_KM)
    & (instituciones_geo["TORRE_CERCANA"].fillna("") != "")
].copy()

nivel3_candidatas = nivel3_candidatas.sort_values("DISTANCIA_TORRE_ESTATAL_CERCANA", ascending=True)
nivel3_candidatas = validar_lineas_vista_con_mde(nivel3_candidatas)
nivel3_candidatas["CONEXION_INICIAL"] = np.where(
    nivel3_candidatas["LINEA_VISTA"],
    "CONEXION DIRECTA VIABLE",
    "REQUIERE REPETIDOR O REVISION TECNICA",
)

nivel3_export = pd.DataFrame({
    "ID_INSTITUCION": nivel3_candidatas["ID"],
    "INSTITUCION": nivel3_candidatas["NOMBRE"],
    "SECTOR": nivel3_candidatas["GRUPO_INSTITUCION"],
    "MUNICIPIO": nivel3_candidatas["MUNICIPIO"],
    "LATITUD": nivel3_candidatas["LATITUD"].round(6),
    "LONGITUD": nivel3_candidatas["LONGITUD"].round(6),
    "TORRE_CERCANA": nivel3_candidatas["TORRE_CERCANA"],
    "DEPENDENCIA_TORRE_ESTATAL": nivel3_candidatas["DEPENDENCIA_TORRE_ESTATAL"],
    "LATITUD_TORRE": nivel3_candidatas["LATITUD_TORRE_ESTATAL"].round(6),
    "LONGITUD_TORRE": nivel3_candidatas["LONGITUD_TORRE_ESTATAL"].round(6),
    "DIST_KM": nivel3_candidatas["DISTANCIA_TORRE_ESTATAL_CERCANA"].round(3),
    "LINEA_VISTA": nivel3_candidatas["LINEA_VISTA"],
    "CONEXION_INICIAL": nivel3_candidatas["CONEXION_INICIAL"],
    "OBSTRUCCION_MAX_M": nivel3_candidatas["OBSTRUCCION_MAX_M"].round(2),
})

ruta_nivel3 = OUTPUT_DIR / "conexiones_nivel3_torres_estatales.csv"
nivel3_export.to_csv(ruta_nivel3, index=False, encoding="utf-8-sig")

print("Archivo generado:", ruta_nivel3)
print("Instituciones Nivel 3 candidatas a conexión inicial:", len(nivel3_export))
print("Candidatas con línea de vista directa:", int(nivel3_export["LINEA_VISTA"].sum()))
display(nivel3_export.head())


Archivo generado: ENTREGABLES_FINALES\conexiones_nivel3_torres_estatales.csv
Instituciones Nivel 3 candidatas a conexión inicial: 755
Candidatas con línea de vista directa: 714


,ID_INSTITUCION,INSTITUCION,SECTOR,MUNICIPIO,LATITUD,LONGITUD,TORRE_CERCANA,DEPENDENCIA_TORRE_ESTATAL,LATITUD_TORRE,LONGITUD_TORRE,DIST_KM,LINEA_VISTA,CONEXION_INICIAL,OBSTRUCCION_MAX_M
0,4570,H.AYUNTAMIENTO - ROSARIO TESOPACO,GOBIERNO,ROSARIO,27.840593,-109.370759,H. Ayuntamiento de Rosario,Telefonía Rural,27.840604,-109.370759,0.001,True,CONEXION DIRECTA VIABLE,-10.17
1,4545,H.AYUNTAMIENTO - ETCHOJOA,GOBIERNO,ETCHOJOA,26.911615,-109.625925,(TZ-45) Torre Palacio Municipal Etchojoa,Seguridad,26.911617,-109.625892,0.003,True,CONEXION DIRECTA VIABLE,-10.17
2,3328,CENTRO DE SALUD RURAL YAVARITOS,SALUD,HUATABAMPO,26.779108,-109.368831,Centro de Salud Rural de Yavaritos,Telefonía Rural,26.779140,-109.368830,0.004,True,CONEXION DIRECTA VIABLE,-10.17
3,4577,H.AYUNTAMIENTO - SANTA ANA,GOBIERNO,SANTA ANA,30.544353,-111.121269,SANTA ANA,Telemax,30.544389,-111.121275,0.004,True,CONEXION DIRECTA VIABLE,-10.17
4,4571,H.AYUNTAMIENTO - SAHUARIPA,GOBIERNO,SAHUARIPA,29.056336,-109.233486,H. Ayuntamiento de Sahuaripa 2,Telefonía Rural,29.056370,-109.233470,0.004,True,CONEXION DIRECTA VIABLE,-10.17


## Entregable 6: Torres propuestas

Para proponer nueva infraestructura se analizan instituciones de **Nivel 4**, es decir, instituciones sin conectividad y sin torre estatal cercana. Se usa `DBSCAN` con distancia Haversine para detectar grupos territoriales con al menos 5 instituciones cercanas entre sí.

Por cada grupo se selecciona una institución representativa mediante el criterio de **medoide**: el punto cuya distancia promedio hacia las demás instituciones del grupo es menor. Ese punto funciona como ubicación sugerida inicial para una torre o nodo de conectividad.

Después se valida la **línea de vista** con el Modelo Digital de Elevación `26_Sonora_r15m_v4.tif`. Para cada propuesta generada por el agrupamiento se compara el perfil del terreno contra la línea recta entre la antena propuesta y la torre estatal cercana, usando una altura de 30 metros para ambas torres.


In [12]:
def seleccionar_medoide(cluster_df):
    coords_rad = np.radians(cluster_df[["LATITUD", "LONGITUD"]].to_numpy())
    distancias = haversine_distances(coords_rad) * EARTH_RADIUS_KM
    promedio = distancias.mean(axis=1)
    maximo = distancias.max(axis=1)
    pos = int(np.argmin(promedio))
    representante = cluster_df.iloc[pos].copy()
    representante["DISTANCIA_PROMEDIO_CLUSTER"] = promedio[pos]
    representante["DISTANCIA_MAXIMA_CLUSTER"] = maximo[pos]
    representante["DISTANCIA_MAXIMA_REAL"] = maximo.max()
    representante["CANTIDAD_INSTITUCIONES"] = len(cluster_df)
    return representante


base_cluster = instituciones_geo[instituciones_geo["IPC"] == "NIVEL 4 (CRITICO)"].copy()
coords_cluster_rad = np.radians(base_cluster[["LATITUD", "LONGITUD"]].to_numpy())

dbscan = DBSCAN(
    eps=RADIO_CLUSTER_KM / EARTH_RADIUS_KM,
    min_samples=MIN_INSTITUCIONES_CLUSTER,
    metric="haversine",
)

base_cluster["CLUSTER"] = dbscan.fit_predict(coords_cluster_rad)
clusters_validos = sorted(c for c in base_cluster["CLUSTER"].unique() if c != -1)

representantes = []
for cluster_id in clusters_validos:
    cluster_df = base_cluster[base_cluster["CLUSTER"] == cluster_id].copy()
    rep = seleccionar_medoide(cluster_df)
    rep["CLUSTER"] = cluster_id
    representantes.append(rep)

torres_propuestas = pd.DataFrame(representantes)

torres_propuestas = torres_propuestas.sort_values("CLUSTER").reset_index(drop=True)


def validar_lineas_vista_con_mde(torres_df, raster_path=FUENTE_MDE):
    """Valida línea de vista con el Modelo Digital de Elevación del INEGI."""
    if not Path(raster_path).exists():
        raise FileNotFoundError(f"No se encontró el MDE: {raster_path}")

    try:
        import rasterio
        proj_data = Path(rasterio.__file__).parent / "proj_data"
        if proj_data.exists():
            os.environ["PROJ_DATA"] = str(proj_data)
            os.environ["PROJ_LIB"] = str(proj_data)
        from rasterio.warp import transform as transformar_coords
    except ImportError as exc:
        raise ImportError("Instala rasterio antes de validar línea de vista: %pip install rasterio") from exc

    resultados = []

    with rasterio.open(raster_path) as mde:
        nodata = mde.nodata

        for _, row in torres_df.iterrows():
            lats = np.linspace(row["LATITUD"], row["LATITUD_TORRE_ESTATAL"], MUESTRAS_LINEA_VISTA)
            lons = np.linspace(row["LONGITUD"], row["LONGITUD_TORRE_ESTATAL"], MUESTRAS_LINEA_VISTA)

            xs, ys = transformar_coords("EPSG:4326", mde.crs, lons.tolist(), lats.tolist())
            muestras = np.array([valor[0] for valor in mde.sample(zip(xs, ys))], dtype="float64")

            if nodata is not None:
                muestras = np.where(muestras == nodata, np.nan, muestras)

            if np.isnan(muestras).any():
                resultados.append({
                    "LINEA_VISTA": False,
                    "OBSTRUCCION_MAX_M": np.nan,
                    "ELEVACION_PROPUESTA_M": np.nan,
                    "ELEVACION_TORRE_ESTATAL_M": np.nan,
                })
                continue

            altura_inicio = muestras[0] + ALTURA_TORRE_PROPUESTA_M
            altura_fin = muestras[-1] + ALTURA_TORRE_ESTATAL_M
            linea_senal = np.linspace(altura_inicio, altura_fin, MUESTRAS_LINEA_VISTA)

            # Se excluyen los extremos porque corresponden a las propias torres.
            obstrucciones = muestras[1:-1] - linea_senal[1:-1] + MARGEN_OBSTRUCCION_M
            obstruccion_max = float(np.max(obstrucciones)) if len(obstrucciones) else 0.0
            linea_vista = obstruccion_max <= 0

            resultados.append({
                "LINEA_VISTA": bool(linea_vista),
                "OBSTRUCCION_MAX_M": obstruccion_max,
                "ELEVACION_PROPUESTA_M": float(muestras[0]),
                "ELEVACION_TORRE_ESTATAL_M": float(muestras[-1]),
            })

    resultados_df = pd.DataFrame(resultados)
    return pd.concat([torres_df.reset_index(drop=True), resultados_df], axis=1)


torres_propuestas = validar_lineas_vista_con_mde(torres_propuestas)
torres_propuestas["CONEXION_BACKHAUL"] = np.where(
    torres_propuestas["LINEA_VISTA"],
    "CONEXION DIRECTA",
    "SUGERIR REPETIDOR",
)

torres_propuestas_export = pd.DataFrame({
    "GRUPO": torres_propuestas["GRUPO_INSTITUCION"],
    "ID": torres_propuestas["CLAVE_UNICA"].where(torres_propuestas["CLAVE_UNICA"] != "", torres_propuestas["ID"]),
    "NOMBRE": torres_propuestas["NOMBRE"],
    "DIST_KM_TORRE_ESTATAL_MAS_CERCANA": torres_propuestas["DISTANCIA_TORRE_ESTATAL_CERCANA"].round(8),
    "LATITUD": torres_propuestas["LATITUD"].round(6),
    "LONGITUD": torres_propuestas["LONGITUD"].round(6),
    "CLUSTER": torres_propuestas["CLUSTER"],
    "DISTANCIA_PROMEDIO_CLUSTER": torres_propuestas["DISTANCIA_PROMEDIO_CLUSTER"].round(9),
    "DISTANCIA_MAXIMA_CLUSTER": torres_propuestas["DISTANCIA_MAXIMA_CLUSTER"].round(9),
    "DISTANCIA_MAXIMA_REAL": torres_propuestas["DISTANCIA_MAXIMA_REAL"].round(9),
    "CANTIDAD_INSTITUCIONES": torres_propuestas["CANTIDAD_INSTITUCIONES"],
    "TORRE_ESTATAL_CERCANA": torres_propuestas["TORRE_CERCANA"],
    "DEPENDENCIA_TORRE_ESTATAL": torres_propuestas["DEPENDENCIA_TORRE_ESTATAL"],
    "LATITUD_TORRE_ESTATAL": torres_propuestas["LATITUD_TORRE_ESTATAL"].round(6),
    "LONGITUD_TORRE_ESTATAL": torres_propuestas["LONGITUD_TORRE_ESTATAL"].round(6),
    "LINEA_VISTA": torres_propuestas["LINEA_VISTA"],
    "CONEXION_BACKHAUL": torres_propuestas["CONEXION_BACKHAUL"],
})

ruta_torres_propuestas = OUTPUT_DIR / "torres_propuestas_actualizado.csv"
torres_propuestas_export.to_csv(ruta_torres_propuestas, index=False, encoding="utf-8-sig")

print("Archivo generado:", ruta_torres_propuestas)
print("Torres propuestas:", len(torres_propuestas_export))
print("Con línea de vista:", int(torres_propuestas_export["LINEA_VISTA"].sum()))
display(torres_propuestas_export.head())

Archivo generado: ENTREGABLES_FINALES\torres_propuestas_actualizado.csv
Torres propuestas: 9
Con línea de vista: 3


,GRUPO,ID,NOMBRE,DIST_KM_TORRE_ESTATAL_MAS_CERCANA,LATITUD,LONGITUD,CLUSTER,DISTANCIA_PROMEDIO_CLUSTER,DISTANCIA_MAXIMA_CLUSTER,DISTANCIA_MAXIMA_REAL,CANTIDAD_INSTITUCIONES,TORRE_ESTATAL_CERCANA,DEPENDENCIA_TORRE_ESTATAL,LATITUD_TORRE_ESTATAL,LONGITUD_TORRE_ESTATAL,LINEA_VISTA,CONEXION_BACKHAUL
0,ESCUELAS,26DAI0003S,TRIBU GUARIJIO NUM. 1,36.199726,27.571436,-108.865089,0,4.474960,16.828941,17.029669,10,CERRO MOYAHUI (T.R.),CFE,27.545293,-109.231119,False,SUGERIR REPETIDOR
1,SALUD,SRIMB000132,CENTRO DE SALUD RURAL SAN BERNARDO,21.528550,27.400337,-108.846972,1,4.914362,11.664667,18.457408,15,LOS-TANQUES,CFE,27.210550,-108.890060,False,SUGERIR REPETIDOR
2,SALUD,SRIMB000185,CENTRO DE SALUD RURAL EL ZAPOTE,21.531351,26.779452,-108.690657,2,8.949487,22.518194,32.994554,15,TAPIZUELAS,CFE,26.663510,-108.864288,True,CONEXION DIRECTA
3,ESCUELAS,26DNM0030F,NUEVA CREACION PREESCOLAR NIÑOS MIGRANTES,11.104182,31.145556,-110.641389,3,0.016801,0.084004,0.084004,5,SANTA CRUZ,Telemax,31.237444,-110.595678,False,SUGERIR REPETIDOR
4,SALUD,SRIMB002104,CENTRO DE SALUD RURAL AGIABAMPO 1,13.169012,26.364980,-109.143338,4,2.904596,6.539380,6.807070,7,Centro de Salud Rural de Venustiano Carranza,Telefonía Rural,26.427719,-109.031198,True,CONEXION DIRECTA


## Resumen final

Con esta libreta se genera un flujo reproducible para pasar de los archivos fuente a los datasets finales de la entrega. Los archivos se guardan en:

`ENTREGABLES_FINALES/`

Los parámetros definidos al inicio permiten ajustar radios de cobertura, tamaño mínimo de cluster, alturas consideradas para línea de vista y criterios técnicos de priorización.

La libreta deja documentado el proceso completo: integración de instituciones, cálculo de distancias, clasificación IPC, selección de candidatas a migración, selección de candidatas a conexión inicial mediante torres estatales existentes, validación de línea de vista y propuesta de nueva infraestructura.
